In [1]:
from langgraph.graph import StateGraph, START, END
from typing import Dict, Any
from pydantic import BaseModel, Field

In [3]:
#워크플로 단계 정의
class WorkflowStep:
    GREETING = "GREETING"
    PROCESSING = 'PROCESSING'

In [8]:
# 그래프 상태 정의
class GraphState(BaseModel):
    name : str = Field(default='', description="사용자 이름")
    greeting : str = Field(default='', description='생성된 인사말')
    processed_message: str = Field(default='', description='처리된 최종 메시지 ')

In [9]:
# 첫 번째 노드 함수
def generate_greeting(state: GraphState) -> Dict[str, Any]:
    name = state.name or "브로"
    greeting = f"하이! {name}!"
    print(f"[generate_greeting] 인사말 생성 : {greeting}")
    return {'greeting' : greeting}


In [10]:
# 두 번째 노드
def process_message(state : GraphState) -> Dict[str, Any]:
    greeting = state.greeting
    processed_message = f"{greeting} 환영!! "
    print(f"[process_message] 최종 메시지 : {processed_message}")
    return {"processed_message" : processed_message}


In [11]:
# 그래프 생성
def create_hello_graph():
    workflow = StateGraph(GraphState)


    # 노드 추가
    workflow.add_node(WorkflowStep.GREETING, generate_greeting)
    workflow.add_node(WorkflowStep.PROCESSING, process_message)


    # 시작점 설정
    workflow.add_edge(START, WorkflowStep.GREETING)


    workflow.add_edge(WorkflowStep.GREETING, WorkflowStep.PROCESSING)
    workflow.add_edge(WorkflowStep.PROCESSING, END)


    app = workflow.compile()


    return app


In [12]:
app = create_hello_graph()
initial_state = GraphState(name = "플레이데이터", greeting="", processed_message='')


final_state = app.invoke(initial_state)


png = app.get_graph().draw_mermaid_png()


with open("./first_graph.png", "wb") as f:
    f.write(png)


[generate_greeting] 인사말 생성 : 하이! 플레이데이터!
[process_message] 최종 메시지 : 하이! 플레이데이터! 환영!! 


In [1]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(model = 'gpt-5-mini', temperature=0.2)
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [2]:
hr_persona= """
당신은 회사 인사 담당자입니다. 아래 내용은 지원자의 이력서입니다.
{resume}
아래 조건을 만족하는 자소서를 판단해서
'통과', '불통과' 중 하나로 분류해주세요. 답변은 반드시 하나의 단어만 출력할 것!


##기본 평가항목
- 질문 항목에 대한 답변으로 적절한가?
- 지원자의 지원동기 등 입사에 대한 열의가 있는가?
- 핵심가치 및 인재상에 부합하는 인재인가?
- 지원 직무에 대한 역량은 갖추었는가?
- 예비 사회인으로서 다양한 경험을 하였는가?
- 팀워크, 리더십, 갈등해결 등 대인관계능력을 갖추었는가?
- 지원 기업에 대한 이해도를 갖추고자 노력하였는가?
- 다양한 변화에 대응할 인문학적 소양 등을 갖추었는가?


##가점요인
- 역량 중심으로 고민을 담아 성의껏 작성하였는가?
- 경험 위주로 근거를 갖고 구체적으로 기술하고 있는가?
- 표현능력(논리력, 문장력 등)이 우수한가?


##감점요인
- 동일 내용을 반복하고 있는가?
- 제시된 분량에 턱없이 모자라는가? (70% 이하)
- 원론적이고 추상적인 내용으로 기술하고 있는가?
- 질문과 무관한 내용이 담겨 있는가?
- 오타 및 부정확한 표현은 없는가?
- 타사 지원서를 복사하여 지원 회사명이 잘못 기재하였는가?
"""


In [3]:
prompt = PromptTemplate(
    template=hr_persona
)


chain = prompt | llm | StrOutputParser()


In [4]:
chain.invoke({'resume' : """최근 챗GPT를 시작으로 AI 시장은 2027년 약 500조 규모로 성장할 것으로 예상되면서,
급격한 정보 처리량의 증가에 대비해 고성능, 고용량 메모리 반도체가 필수가 되었습니다.
삼성전자는 업계 최초로 HKMG 공정을 적용시켜
기존 공정 대비 저전압인 1.1V에서도 고성능을 구현하고
전력 소모를 13% 감소시킨 DDR5을 양산하며 메모리 초격차를 유지하고,
12나노급 32Gb DDR5 D램은 TSV 공정 없이 양산함으로써
제조기술력까지 입증하고 있습니다.
HKMG는 두께가 얇아짐에 따라 증가하는 누설 전류 이슈를 해결할 수 있는 해결책이지만,
공정 시간과 비용이 증가한다는 단점이 있습니다.
‘진공과 플라즈마’, ‘첨단박막공학’과목을 수강하고,
학부연구생과 캡스톤디자인을 하며 ‘마그네트론 스퍼터’,
‘HFCVD’ 같은 증착 장비를 사용해본 경험을 기반으로
high-K 증착 속도의 최적화를 달성하는데 기여하고 싶습니다.
공정 최적화 역량을 갖춘 재료공학도로서 금속-반도체 접합 특성을 이해하여
메모리사업부의 Deposition 공정 기술을 고도화하겠습니다.
장기적으로는 소재 특성과 공정 간의 유기적 사고를 통해, 공정 이슈를 분석하고
적절한 Action item을 제시하는 Deposition 엔지니어로 성장하겠습니다.






2. 본인의 성장과정을 간략히 기술하되 현재의 자신에게 가장 큰 영향을 끼친 사건,
인물 등을 포함하여 기술하시기 바랍니다. (※작품 속 가상인물도 가능)
1500자 이내 (영문작성 시 3000자)


[서핑동아리 ‘WAVE’를 만들다]
2년 동안 서핑동아리 'WAVE'를 정식 중앙동아리로 만드는 목표를 세우고 성취한 경험이 있습니다.
2021년 당시 서핑은 ‘비싸고 위험하다’라는인식이 강했습니다.
인식을 변화시키고 또래들과 함께 서핑을 즐기고자
저를 포함한 5명이 모임을 만들었습니다.
서핑 대여점과 제휴를 맺어 대여비를 기존 대비 43% 할인 시켰습니다.
비싼 가격을 줄이기 위해서 서핑 대여점과 제휴를 맺는 것이 필수였습니다.
 하지만 처음 제안은 모임 인원수가 적어 거절당했습니다.
그래서 인원 문제를 해결하기 위해 서핑하며 만난 학생들에게 제안하고,
에브리타임에 홍보하여 3개월 동안 50명의 부원을 모집했습니다.
또한 인원을 모으며 제휴를 맺었을 때의 장점을 제휴업체 입장에서 생각하려 노력했습니다.
성수기를 대비한 강사 보충, ‘WAVE’로부터 창출될 홍보 효과를 규모의 경제 측면으로 설명하고 제안하여
결국 제휴를 성공적으로 맺었습니다.
‘위험하다’라는 인식을 바꾸기 위해서는 시각적 자료가 필요했습니다.
이를 위해 ‘WAVE’ 인스타그램 계정을 개설해 ‘광안리 SUPrise 대학교 대항전’에 참여하며
안전하게 훈련하는 과정 영상을 업로드했습니다.
여기서 저는 훈련부장으로서 제휴업체와 부원 간의 일정을 조율하고 훈련을 진행했습니다.
그리고 나머지 회장단은 인스타그램 관리, 영상 촬영, 편집을 배분하여 진행했습니다.
결과적으로 모두의 노력이 결실을 맺어 대회 3등 수상과 ‘생각보다 위험하지 않다’라는
인식의 변화를 만들 수 있었습니다.
기존에 없던 시스템을 구축하고,
혼자서는 결코 성공하지 못했을 ‘WAVE’를 성장시키며
‘같이하면 불가능한 것도 가능하다.’는 교훈을 배웠습니다.


[CVD 가스 유량 최적화 : 효율적인 인수인계]
노션을 이용해 효율적으로 프로젝트를 진행함으로써 한국표면공학회에서 ‘BDD를 활용한
육상양식장 폐수처리용 고효율 불용성 양극전극 개발’이란 주제로
포스터 발표 우수상을 받았습니다.
학회 일정상의 문제와 전구체인
 B, C, H 공급 가스의 최적 유량을 찾기 위해 많은 실험이 필요했고
한 번 실험하는데 8시간 넘게 소요되는 문제가 있었습니다.
공정 시간을 줄이는 법과 시간을 효율적으로 사용하는 방법을 고민했습니다.
공정 시간의 경우, BDD의 수처리 효율을 고려해 2㎛ 이상 증착시켜야 하기 때문에
단시간에 줄이는 것이 불가능했습니다.
그래서 시간을 효율적으로 사용하기 위해 3교대 실험을 제안했습니다.
하지만 교대를 하게 되니
어떤 조건으로 실험하는지 모르는 상황이 발생해서 인수인계가 중요해졌습니다.
이를 해결하기 위해 노션에 실험 조건을 정리하고,
대표자 한 명이 다음 교대 팀에게 진행 상황을 보고하는 시스템을 도입했습니다.
이 방식으로 촉박한 시간 속에서 B/C ratio 2500ppm이란
최적의 조건으로 수처리 분석까지 완료하여
 학회에서 좋은 결과를 만들었습니다.
위의 경험을 살려서 삼성전자의 공정기술 직무에서도 소통을 통해
효율적으로 프로젝트를 진행하는 주체적인 엔지니어로 성장하겠습니다.






3. 최근 사회 이슈 중 중요하다고 생각되는 한 가지를 선택하고
이에 관한 자신의 견해를 기술해 주시기 바랍니다.
1000자 이내 (영문작성 시 2000자)


[과학기술 발전을 통한 탄소중립 실현]
‘파리 협정 2015’ 이후, 2℃ 이하의 지구 평균온도 상승 폭을 유지하기 위해 세계적으로 노력하고 있습니다.
하지만, 최근 연구 결과에 따르면,
현재와 같은 감축 속도로는 2033년에서 2035년 사이에 1.5℃가 증가하여
10년 안에 목표가 좌절될 가능성이 매우 높고,
지금부터 모든 인류가탄소 배출을 하지 않아도
2065년 이전에 2℃ 상승할 확률이 80%입니다.
이에 따라 폭염, 가뭄, 홍수 등 극단적인 기상 현상이 발생하고 생태계의 균형이 깨질 수 있습니다.
게다가 향후 EU의 탄소국경세 혹은 글로벌 기업들의 RE100 가입 및 기준이 강화될 경우
기업의 브랜드 가치 저하 및 실적 악화로 이어질 수있어,
특히나 많은 양의 전력을 소비하는 종합반도체 기업은 보다
강도 높은 선제적인 대응이 필요합니다.
삼성전자의 경우, 2027년 해외 사업장과 DX 부문 RE100 달성을 선언하고
반도체 업계 최초로 CCUS를 상용화하기 위해 탄소 포집 연구소를 설립해
기술 발전을 통한 환경 난제 해결에 도전하고 있습니다.
이는 반도체 업계의 탄소 배출 문제를 근본적으로 해결할 수 있는 시발점이 될 것입니다.
또한 업계 최초로 12나노급 32Gb DDR5 D램을 개발하며 동일 128Gb 모듈 기준,
16Gb D램을 탑재한 모듈 대비 약 10% 소비 전력을 개선시켰습니다.
앞으로 데이터를 저장하고 처리하는 과정에서 많은 에너지가 소모될 것입니다.
IT/데이터 시대에서 삼성전자의 반도체는 저전력, 고효율 메모리 제품 생산을 통한
전력 및 운영비용 감소로 탄소중립에도 기여할 것입니다.
결국, 탄소중립을 위해서는 탄소포집기술, 소비전력 효율을 개선,
친환경 에너지의 사용 같은 과학기술을 발전시키는 근본적인 문제해결과정이 필요합니다.
삼성전자는 앞으로도 사회적, 환경적 측면에서 함께 공존할 수 있는 과학기술을 고민해 나감으로써
글로벌 기업으로서 위상을 높여갈 수 있을 것입니다.






4. 지원한 직무 관련 본인이 갖고 있는 전문지식/경험(심화전공, 프로젝트, 논문, 공모전 등)을 작성하고,
이를 바탕으로 본인이 지원 직무에 적합한 사유를 구체적으로 서술해 주시기 바랍니다.
1000자 이내 (영문작성 시 2000자)


공정기술 직무에서는 공정 변수가 구조에 미치는 영향을 이해하고
최적화된 공정을 하는 것이 중요합니다.
[Al/Ti 200nm 다층 박막 형성 및 나노 경도 측정] ‘마그네트론 스퍼터’를 이용해
 Al/Ti 각각 100nm씩 200nm 다층 박막을 형성하여 나노 경도를 측정했습니다.
해당 연구는 스퍼터를 처음 셋업하면서 각 타겟의 증착 속도를 구하는 것과
Si-wafer에 증착된 물질의 미세구조를 이용해
 박막의 경도를 향상하는 두 가지 목표가 있었습니다.
박막의 두께에 따라 바뀌는 특성 때문에 형상두께는 200nm로 고정하고,
‘파센 법칙’과 ‘Thornton Zone Model’을 이용해서
공정변수를 인가전압과 작업압력, 기판 온도로 설정했습니다.
변수가 박막에 주는 영향을 확인하기 위해 경향성 test를 진행했습니다.
1. 기판 온도는 Al의 저융점 특성 때문에 600도 이상 올리는 것이 불가능해
Ti의 주상구조가 나타나는 온도인 400도로 설정했습니다.
2. 인가전압의 경우, 350W 이상일 때
Ar 이온의 에너지가 강해 스퍼터링율이 감소하여
증착 속도가 느려지는 현상으로 인해 300W로 설정했습니다.
3. 압력의 경우 증착 속도는 7mtorr에서 가장 높았지만, 결정성은 3mtorr에서 가장 좋았기 때문에
3mtorr로 설정했습니다.
총 76번의 실험과 시편 분석을 한 결과
‘인가전압 300W, 작업압력 3mTorr, 기판 온도 400도’ 조건으로
Al, Ti 각각 25.1, 19.8nm/min이란 증착속도를 구할 수 있었습니다.
구한 증착 속도로 본 실험에서 200nm의 다층 박막을
오차범위 5% 이내로 형성했고 나노 경도를 측정했습니다.
50mN 하중을 기준으로, 박막의 결정성이 존재할 때,
비정질일 때보다 나노 경도가 약 1300Hv로 31% 향상됐습니다.
이 경험을 통해 공정 최적화를 위한 변수에 대한 이해도를 높일 수 있었습니다.
 Depo 공정기술 엔지니어로서 공정 최적화를 통한 안정적 수율확보에 기여하겠습니다.


대학생 대외활동 공모전 채용 사이트 링커리어 https://linkareer.com/"""})


'통과'

In [5]:
from typing import Dict, Any, Literal, List
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field  


In [6]:
class HRBotState(BaseModel):
    cover_letter : str = Field(default='', description="사용자가 입력한 자기소개서")
    result : str = Field(default='', description='자기소개서의 결과 값')
    response : str = Field(default='', description='최종 응답 메시지')
def analyze_hr(state: HRBotState ) -> Dict[str, Any]:
    cover_letter = state.cover_letter
    rt = chain.invoke({'resume' : cover_letter})
    return {'result' : rt}
def pass_response(state: HRBotState) -> Dict[str, Any]:
    return {'response' : '다음 진행 상태를 확인하세요'}
def failed_response(state : HRBotState) -> Dict[str, Any]:
    hr_persona= """
    당신은 회사 인사 담당자입니다. 아래 내용은 지원자의 이력서입니다. 통과하지 못한 이유를 자세히 설명할 것
    {resume}
    아래 조건을 만족하는 자소서를 판단할 때 사용한 규칙이다.


    ##기본 평가항목
    - 질문 항목에 대한 답변으로 적절한가?
    - 지원자의 지원동기 등 입사에 대한 열의가 있는가?
    - 핵심가치 및 인재상에 부합하는 인재인가?
    - 지원 직무에 대한 역량은 갖추었는가?
    - 예비 사회인으로서 다양한 경험을 하였는가?
    - 팀워크, 리더십, 갈등해결 등 대인관계능력을 갖추었는가?
    - 지원 기업에 대한 이해도를 갖추고자 노력하였는가?
    - 다양한 변화에 대응할 인문학적 소양 등을 갖추었는가?


    ##가점요인
    - 역량 중심으로 고민을 담아 성의껏 작성하였는가?
    - 경험 위주로 근거를 갖고 구체적으로 기술하고 있는가?
    - 표현능력(논리력, 문장력 등)이 우수한가?


    ##감점요인
    - 동일 내용을 반복하고 있는가?
    - 제시된 분량에 턱없이 모자라는가? (70% 이하)
    - 원론적이고 추상적인 내용으로 기술하고 있는가?
    - 질문과 무관한 내용이 담겨 있는가?
    - 오타 및 부정확한 표현은 없는가?
    - 타사 지원서를 복사하여 지원 회사명이 잘못 기재하였는가?
    """
    prompt = PromptTemplate(
        template=hr_persona
    )
    chain2 = prompt | llm | StrOutputParser()


    rt = chain2.invoke({'resume' :state.cover_letter })
    return {'response' : rt}


def route_by_hr(state : HRBotState) -> Literal['pass', 'fail']:
    result = state.result
    print(f"라우팅 : {result}")


    if result == "통과":
        return "pass"
    else:
        return "fail"
def create_hr_bot_graph():
    workflow = StateGraph(HRBotState)


    workflow.add_node("analyze_hr", analyze_hr)
    workflow.add_node('pass', pass_response)
    workflow.add_node('fail', failed_response)


    workflow.add_edge(START, 'analyze_hr')
    # 라우팅
    workflow.add_conditional_edges(
        'analyze_hr',
        route_by_hr,
        {
            'pass' : "pass",
            "fail" : "fail"
        }
    )


    workflow.add_edge("pass", END)
    workflow.add_edge("fail", END)


    return workflow.compile()
app = create_hr_bot_graph()
cover_letter = """ 1) 성장괴정
저는 2001년 2월 막내로 태머냤습니다. 저의 청소년시절은 그리 밝은 추역
은 별로 없습니다. 부모님미 유년기때의 마음의 샴처로 인해서 서로 미해하지 못하고
오해로 인해서 저의 학장시절은 그리 행복한 추먹은 없습니다. 그러나 가정의 불화로
인해서 방황하며 힘들머 하고 있을 때 저메계 힙미 되머주신 교회 전도사님미 계셨습니
다. 그분의 위로와 격려로 인해서 고등학교를 졸업 후 직장을 다니면서 꿈을 가지계 되
였고, 직장을 퇴사한 후 대전에 있는 침례신학대학교메서 사회복지학과를 미수하고, 가
속간의 화해를 위해서 부모님곁메 있었습니다. 그것미 사회복지를 배문자로서 그리고
저의 꿈을 위한 사회복지사로서의 시작미있습니다.
2) 성격의 장. 단점
저의 장점과 단정은 같은 것입니다. 먼저 저의 단정은 무슨일을 하는데 있머서 생각
미 많고 너무 신중하다는 것입니다. 그로 인해서 결정을 내리는데 시간미 걸린다는 것
입니다. 그에 반대 장점은 결정을 내련 것에 대해서 나 자신을 믿고 그 일에 대해서 책
임을 지고 행한다는 것입니다.
3) 입사 지원동기
저는 사회복지 문먀메 관심미 많습니다. 특히 저의 성장 과정을 통해서 가족의 화목
미 얼마나 중요함을 매달게 되었습니다. 가족구성들의 각자 자기의 위치에서 누리며 살
아가며 서로 격려하|주며 살아가는 것미 건강한 가족미라 생각하며, 건강한 가족만미 건
강한 사회를 미끌머간다고 생각햘니다.
또한 대학재학시절 경기북부멍마일시보호소메서 자원봉사를 대본 경험미 있습니다.
그때 멍아들미 머떻계 보호소에 들머오는지를 보았고, 머떻계 AH로문 가정을 만나서 가
는 모습을 보면서 저의 작은 손길미 마미들메계 도움미 되었으면 하는 마음으로 지원하
계 되었습니다.
4) 장래의 희망 또는 포부
저의 장래의 희망은 위기 가정에 도움을 주는 사회복지사가 되는 것입니다. 제가 -
을 수 없는 상황께서 웃으면서 살아가고 있는 것처럼 단 한몀미라도 더 웃으며 실파가
도록 도와 줄 것입니다."""
state = HRBotState(cover_letter=cover_letter)
app.invoke(state)




라우팅 : 불통과


{'cover_letter': ' 1) 성장괴정\n저는 2001년 2월 막내로 태머냤습니다. 저의 청소년시절은 그리 밝은 추역\n은 별로 없습니다. 부모님미 유년기때의 마음의 샴처로 인해서 서로 미해하지 못하고\n오해로 인해서 저의 학장시절은 그리 행복한 추먹은 없습니다. 그러나 가정의 불화로\n인해서 방황하며 힘들머 하고 있을 때 저메계 힙미 되머주신 교회 전도사님미 계셨습니\n다. 그분의 위로와 격려로 인해서 고등학교를 졸업 후 직장을 다니면서 꿈을 가지계 되\n였고, 직장을 퇴사한 후 대전에 있는 침례신학대학교메서 사회복지학과를 미수하고, 가\n속간의 화해를 위해서 부모님곁메 있었습니다. 그것미 사회복지를 배문자로서 그리고\n저의 꿈을 위한 사회복지사로서의 시작미있습니다.\n2) 성격의 장. 단점\n저의 장점과 단정은 같은 것입니다. 먼저 저의 단정은 무슨일을 하는데 있머서 생각\n미 많고 너무 신중하다는 것입니다. 그로 인해서 결정을 내리는데 시간미 걸린다는 것\n입니다. 그에 반대 장점은 결정을 내련 것에 대해서 나 자신을 믿고 그 일에 대해서 책\n임을 지고 행한다는 것입니다.\n3) 입사 지원동기\n저는 사회복지 문먀메 관심미 많습니다. 특히 저의 성장 과정을 통해서 가족의 화목\n미 얼마나 중요함을 매달게 되었습니다. 가족구성들의 각자 자기의 위치에서 누리며 살\n아가며 서로 격려하|주며 살아가는 것미 건강한 가족미라 생각하며, 건강한 가족만미 건\n강한 사회를 미끌머간다고 생각햘니다.\n또한 대학재학시절 경기북부멍마일시보호소메서 자원봉사를 대본 경험미 있습니다.\n그때 멍아들미 머떻계 보호소에 들머오는지를 보았고, 머떻계 AH로문 가정을 만나서 가\n는 모습을 보면서 저의 작은 손길미 마미들메계 도움미 되었으면 하는 마음으로 지원하\n계 되었습니다.\n4) 장래의 희망 또는 포부\n저의 장래의 희망은 위기 가정에 도움을 주는 사회복지사가 되는 것입니다. 제가 -\n을 수 없는 상황께서 웃으면서 살아가고 있는 것처럼 단 한몀미라도 더 웃으며 실파